# jobs

> One stored row for each background job: what runs, what ended, and where to see it. The activity chip, the event stream and the Mac app read these rows.

abhyasa does slow work in the background: it looks for sources, makes a walk from a link or a goal, files a folder, plans a course,
and writes the notebook of a step. Each such job has one row here. A row has a `kind`, a `state` (`running`, `done`, `failed`,
`stopped` or `closed`), a `said` line, the `start` and `end` times, a `title`, and a `url`: the page of its object.

The rows are in vishalakshi's job `Ledger` (`Vault.ledger`), a table in the vault file, so they survive a restart. This module is thin
glue over it, with the names that the blocks use. The pool future of a job stays in memory; the row is the truth for the pages. At a
start, each row that still says `running` becomes `closed`: no job lives across a restart.

In [ ]:
#| default_exp jobs

In [ ]:
#| hide
import os, tempfile
os.environ.setdefault('LITESEARCH_CACHE', tempfile.mkdtemp(prefix='ls-'))   # and litesearch does not wait on the shared lock of another run
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # the settings go to a temporary folder, not to your real settings

In [ ]:
#| hide
from fastcore.test import *

The rows live in the vault file, next to the concepts and the walks.

In [ ]:
#| export
import time
from fastcore.utils import *
from abhyasa.store import Learning, _src, log

## The row

`ENDED` are the states of a job that does not run. `FRESH` is how long an ended job shows in the activity chip, until you open its
page. `KEEP` is how long an ended row stays at all: the ledger deletes it at the next start of a job.

In [ ]:
#| export
from vishalakshi.jobs import ENDED
FRESH = 6 * 3600   # seconds that an ended job shows in the activity chip, until you open its page
KEEP = 2 * 86400   # seconds that an ended row stays, then it goes

abhyasa 0.0.15 kept all the rows in the meta of one document (`abhyasa:jobs:all`). The first read of the ledger moves those rows to it
one time, and forgets the document. A row that said `running` is `closed`, as its job did not live across the restart. A row that
abhyasa cannot read is left out: an old row never stops the app.

In [ ]:
#| export
OLD = _src('jobs', 'all')   # the one document of rows of abhyasa 0.0.15

def _move_old(lr:Learning, lg) -> int:
    "Move the rows of the old jobs document to the ledger `lg`, one time, then forget the document. Return the count moved."
    if (d := lr.v.by_source(OLD)) is None: return 0
    n, m = 0, d['meta'] if isinstance(d['meta'], dict) else {}
    for k, r in (m.get('rows') if isinstance(m.get('rows'), dict) else {}).items():
        try:
            if not isinstance(r, dict) or lg.get(str(k)) is not None: continue
            r = {x: y for x, y in r.items() if x != 'key'}
            if r.get('state') == 'running': r.update(state='closed', end=time.time())
            lg.put(str(k), **r); n += 1
        except Exception as e: log.warning('abhyasa left out an old job row %r: %r', k, e)
    lr.v.forget(d['id'])
    return n

def ledger(lr:Learning):
    "The job ledger of the vault of `lr` (`Vault.ledger`). The first read moves the rows of abhyasa 0.0.15 to it."
    lg = lr.vault.ledger
    if not getattr(lg, '_abhyasa_moved', False):
        lg._abhyasa_moved = True
        try: _move_old(lr, lg)
        except Exception as e: log.warning('abhyasa could not move the old job rows: %r', e)
    return lg

@patch
def jobs(self:Learning) -> dict:
    "Each job row by its key, as an `AttrDict` with its `key`."
    return {r.key: r for r in ledger(self).rows()}

@patch
def job(self:Learning, key:str) -> AttrDict|None:
    "The job row `key`, or None."
    return ledger(self).get(key)

A change merges into one row. The ledger counts each change in `version`, also a change from another process.

In [ ]:
#| export
@patch
def put_job(self:Learning, key:str, **kv) -> AttrDict:
    "Merge `kv` into the job row `key` and save it."
    return ledger(self).put(key, **kv)

def job_version(lr:Learning) -> int:
    "A count that goes up at each change of a row."
    return ledger(lr).version

## Start, end, and a pool job

`job_start` writes a running row. `obj` is the object of the job, such as `walk:<id>` or `course:<id>`; its page marks the end as
seen. `quiet` keeps a job out of the chip, such as the research of a side walk of a talk. `job_end` closes it.

In [ ]:
#| export
def job_start(lr:Learning, key:str, kind:str, title:str='', said:str='', url:str='', obj:str='', quiet:bool=False) -> AttrDict:
    "Note that the job `key` of `kind` runs now, with its `title`, its `said` line, the `url` of its page and its object `obj`. Ended rows older than `KEEP` go."
    return ledger(lr).start(key, kind, title=title, said=said, url=url, obj=obj, quiet=quiet)

def job_end(lr:Learning, key:str, state:str='done', why:str='', **kv) -> AttrDict|None:
    "Note that the job `key` ended now in `state`, with `why` for a failure. `kv` can change its `said`, `url` or `obj`. None when there is no row."
    return ledger(lr).end(key, state, why=why, **kv)

`track` gives a pool future its row: running now, then `done` or `failed` when it ends. `ok` maps the result to more fields of the row,
such as the link of the walk that the job made, or `state='failed'` for a job that ended with nothing. `fut` can also be a function
that submits the job: `track` calls it after the row starts, so a fast job never writes to a row that is not there yet. A row that ended before its future, such as a job that you stopped, keeps its state.

In [ ]:
#| export
def track(lr:Learning, key:str, fut, kind:str, ok=None, why=None, **kw):
    """Start the row `key` of the pool future `fut` (or of the future that the function `fut` gives), and end it when the future ends:
    `ok(result)` gives more fields (and can set `state`), `why(error)` the line of a failure. A row that ended first stays. Return the future."""
    return ledger(lr).track(kind, fut, key=key, ok=ok, why=why or (lambda e: 'It did not work.'), **kw)

A job that runs long says what it does now. `job_said` writes a new line to a running row. `Said` is the state of one pool job,
and a new `said` goes to its row too, so a page that reads the row sees the step.

In [ ]:
#| export
def job_said(lr:Learning, key:str, said:str) -> bool:
    "Write the line `said` to the row `key` while its job runs. A row that cannot be written never breaks the job. Whether it changed."
    try: return ledger(lr).say(key, said)
    except Exception: return False

class Said(AttrDict):
    "The state of one pool job, for its card; `update(said=…)` also writes the line to the row `key` of `lr`."
    def __init__(self, lr:Learning, key:str, **kw): super().__init__(**kw); self.__dict__.update(_lr=lr, _key=key)
    def update(self, *a, **kw):
        super().update(*a, **kw)
        if 'said' in kw: job_said(self._lr, self._key, kw['said'])

## Seen, closed, and the activity

Opening the page of an object marks its ended jobs as seen. At a start of the app, a row that says `running` has no job any more:
it becomes `closed`.

In [ ]:
#| export
def seen_jobs(lr:Learning, obj:str) -> int:
    "Mark each ended job of the object `obj` as seen. Return the count that changed."
    return ledger(lr).seen(obj)

def close_jobs(lr:Learning) -> int:
    "At a start: each row that says `running` becomes `closed`, as its job did not live across the restart. Return the count."
    lg = ledger(lr)
    ks = lg.rows(state='running').attrgot('key')
    for k in ks: lg.end(k, 'closed')
    return len(ks)

`activity` is what the chip lists: each job that runs, then each ended job that is `done`, `failed` or `closed`, not seen, and
less than `FRESH` old. A job that you stopped is not news. A quiet job never shows.

In [ ]:
#| export
def activity(lr:Learning, now:float=None) -> L:
    "The rows for the activity chip: the running jobs, newest first, then the ended jobs that are new to you."
    return ledger(lr).activity(fresh=FRESH, now=now)

WORD = dict(done='Ready', failed='Failed', closed='Stopped')

def chip_words(rows) -> str:
    "The words of the chip: `2 running` while a job runs, else `Ready: {title}`, `Failed: {title}` or `Stopped: {title}`, with the count of the others."
    if not rows: return ''
    if (n := sum(r.state == 'running' for r in rows)): return f'{n} running'
    r = rows[0]
    return f"{WORD.get(r.state, 'Ready')}: {r.get('title') or 'a job'}" + (f' and {len(rows) - 1} more' if len(rows) > 1 else '')

## Tests

A temporary vault for the tests.

In [ ]:
from abhyasa.store import temp_learning

A job starts with a running row and no end; the row is in the chip.

In [ ]:
lr = temp_learning()
r = job_start(lr, 'walk:w1:research', 'research', title='Softmax', said='Finding sources', url='/w/w1#sources', obj='walk:w1')
test_eq((r.state, r.end, r.key), ('running', None, 'walk:w1:research'))
test_eq([x.key for x in activity(lr)], ['walk:w1:research'])
test_eq(chip_words(activity(lr)), '1 running')

Its end is news until you open the walk.

In [ ]:
v = job_version(lr)
job_end(lr, 'walk:w1:research', said='2 of 3 ideas have a source')
assert job_version(lr) > v
test_eq(chip_words(activity(lr)), 'Ready: Softmax')
test_eq(seen_jobs(lr, 'walk:w1'), 1)
test_eq(activity(lr), [])

An old end and a quiet job do not show; a job that you stopped is not news.

In [ ]:
job_start(lr, 'k2', 'build', title='Old'); job_end(lr, 'k2')
test_eq(activity(lr, now=time.time() + FRESH + 1), [])
job_start(lr, 'k3', 'research', title='Talk', quiet=True)
job_start(lr, 'k4', 'research', title='Mine'); job_end(lr, 'k4', 'stopped')
test_eq([x.key for x in activity(lr)], ['k2'])   # k2 is new now; k3 is quiet and k4 was stopped
test_eq(job_end(lr, 'nope'), None)

A restart closes each running row, and the chip says so.

In [ ]:
job_start(lr, 'k5', 'build', title='Diffusion')
test_eq(close_jobs(lr), 2)   # k3 and k5
test_eq(lr.job('k5').state, 'closed')
test_eq(chip_words(activity(lr)), 'Stopped: Diffusion and 1 more')

`track` ends the row of a pool future: `done` with the fields of `ok`, or `failed` with the line of `why`. `ok` can also make it `failed`.

In [ ]:
import threading
from concurrent.futures import ThreadPoolExecutor

In [ ]:
pool = ThreadPoolExecutor(1)
track(lr, 'b1', pool.submit(lambda: 'w9'), 'build', ok=lambda jid: dict(url=f'/w/{jid}', obj=f'walk:{jid}'), title='A link').result()
pool.shutdown(wait=True)
test_eq((lr.job('b1').state, lr.job('b1').url, lr.job('b1').obj), ('done', '/w/w9', 'walk:w9'))
pool = ThreadPoolExecutor(1)
f = track(lr, 'b2', pool.submit(lambda: 1 / 0), 'build', why=lambda e: 'No walk.', title='Bad')
pool.shutdown(wait=True)
test_eq((lr.job('b2').state, lr.job('b2').why), ('failed', 'No walk.'))
pool = ThreadPoolExecutor(1)
track(lr, 'b4', lambda: pool.submit(lambda: None), 'demo', ok=lambda d: None if d else dict(state='failed', why='No demo.'), title='Late').result()
pool.shutdown(wait=True)
test_eq((lr.job('b4').state, lr.job('b4').why), ('failed', 'No demo.'))   # a function that submits; `ok` sets the state
pool, gate = ThreadPoolExecutor(1), threading.Event()
f = track(lr, 'b5', lambda: pool.submit(gate.wait, 5), 'sort', title='Sort')
job_end(lr, 'b5', 'stopped'); gate.set(); f.result(); pool.shutdown(wait=True)
test_eq(lr.job('b5').state, 'stopped')   # you stopped it: its late end does not make it done

A new line of a running job goes to its row.

In [ ]:
job_start(lr, 'b3', 'build', title='Paper', said='Reading the paper…')
st = Said(lr, 'b3', said='Reading the paper…')
st.update(said='Reading 2 of 3 pages…')
test_eq((st.said, lr.job('b3').said), ('Reading 2 of 3 pages…', 'Reading 2 of 3 pages…'))
test_eq((job_said(lr, 'b3', 'Reading 2 of 3 pages…'), job_said(lr, 'nope', 'x')), (False, False))   # no change, no row

An ended row older than `KEEP` goes at the next start of a job.

In [ ]:
lr.put_job('b2', end=time.time() - KEEP - 1)
job_start(lr, 'b6', 'build', title='Next')
assert lr.job('b2') is None and lr.job('b3') is not None

An old jobs document of abhyasa 0.0.15 moves to the ledger at the first read. Its running row is `closed`, its other rows keep
their state, and a row that is not a dict is left out. The document goes, so the move is one time.

In [ ]:
old = temp_learning()
rows = {'walk:w7:research': dict(kind='research', state='running', title='Old walk', start=time.time() - 60, obj='walk:w7'),
        'build:b7': dict(kind='build', state='done', title='Old link', start=time.time() - 90, end=time.time() - 30, url='/w/w7', seen=False),
        'bad': 'not a row'}
old.v.add('The background jobs of abhyasa.', title='abhyasa jobs', source=OLD, kind='jobs', meta=dict(rows=rows))
test_eq((old.job('walk:w7:research').state, old.job('build:b7').url, old.job('bad')), ('closed', '/w/w7', None))
test_eq(sorted(old.jobs()), ['build:b7', 'walk:w7:research'])
assert old.v.by_source(OLD) is None   # the old document is gone
test_eq(chip_words(activity(old)), 'Stopped: Old walk and 1 more')   # the closed row ended last

A jobs document with meta that is not JSON of rows does not stop the app: it has no rows, and it goes.

In [ ]:
odd = temp_learning()
odd.v.add('The background jobs of abhyasa.', title='abhyasa jobs', source=OLD, kind='jobs', meta=dict(rows='not a dict'))
test_eq(odd.jobs(), {})
assert odd.v.by_source(OLD) is None

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()